In [ ]:
import os
import sys
sys.path.append("/home/justin/code/Manipulator-Software")
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import glob
import copy
import open3d as o3d


import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Increase the embedding limit (set to 50 MB)
plt.rcParams['animation.embed_limit'] = 500

from tapnet.torch import tapir_model
from tapnet.utils import transforms
from tapnet.utils import viz_utils
import torch
import torch.nn.functional as F


from utils.perception.camera import convert_pixel_to_world_batch, convert_pixel_to_world
from utils.visualization.bounding_box import draw_posed_3d_box, draw_xyz_axis

In [ ]:
data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/edamame_box"
checkpoint_path = "/home/justin/code/Manipulator-Software/perception/tapnet/checkpoints/causal_bootstapir_checkpoint.pt"

resize_h = 256
resize_w = 256

cam2world = np.eye(4)

pcd_radius_outlier_removal = True
pcd_radius_outlier_removal_radius = 0.2
pcd_stat_outlier_removal = True
pcd_stat_outlier_removal_nb_neighbors = 20
pcd_stat_outlier_removal_std_ratio = 2.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_points = 30





In [ ]:


# Function to load RGB images
def load_rgb_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load RGB images from the /rgb subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.jpg', '.jpeg', '.png'])
    
    Returns:
        List[np.ndarray]: List of RGB images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png']
    
    rgb_folder = os.path.join(folder_path, 'rgb')
    if not os.path.exists(rgb_folder):
        raise FileNotFoundError(f"RGB folder not found: {rgb_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(rgb_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            img = cv2.imread(file_path)
            if img is not None:
                # Convert BGR to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img_rgb)
    
    return images

# Function to load depth images
def load_depth_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load depth images from the /depth subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /depth subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.tiff', '.tif'])
    
    Returns:
        List[np.ndarray]: List of depth images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.png', '.tiff', '.tif']
    
    depth_folder = os.path.join(folder_path, 'depth')
    if not os.path.exists(depth_folder):
        raise FileNotFoundError(f"Depth folder not found: {depth_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(depth_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load depth image (usually 16-bit)
            img = cv2.imread(file_path, cv2.IMREAD_ANYDEPTH)
            if img is not None:
                images.append(img)
    
    return images

# Function to load mask images
def load_mask_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load mask images from the /masks subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /masks subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.jpg', '.jpeg'])
    
    Returns:
        List[np.ndarray]: List of mask images as numpy arrays (binary or grayscale)
    """
    if file_extensions is None:
        file_extensions = ['.png', '.jpg', '.jpeg']
    
    masks_folder = os.path.join(folder_path, 'masks')
    if not os.path.exists(masks_folder):
        raise FileNotFoundError(f"Masks folder not found: {masks_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(masks_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load mask as grayscale
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    
    return images

# Function to load all image types together
def load_all_images(folder_path: str) -> Dict[str, List[np.ndarray]]:
    """
    Load RGB, depth, and mask images from the specified folder structure.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[np.ndarray]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of images
    """
    result = {}
    
    try:
        result['rgb'] = load_rgb_images(folder_path)
        print(f"Loaded {len(result['rgb'])} RGB images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['rgb'] = []
    
    try:
        result['depth'] = load_depth_images(folder_path)
        print(f"Loaded {len(result['depth'])} depth images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['depth'] = []
    
    try:
        result['masks'] = load_mask_images(folder_path)
        print(f"Loaded {len(result['masks'])} mask images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['masks'] = []
    
    return result

# Function to get file paths without loading images
def get_image_paths(folder_path: str) -> Dict[str, List[str]]:
    """
    Get file paths for RGB, depth, and mask images without loading them.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[str]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of file paths
    """
    result = {}
    
    # RGB paths
    rgb_folder = os.path.join(folder_path, 'rgb')
    if os.path.exists(rgb_folder):
        rgb_files = []
        for ext in ['.jpg', '.jpeg', '.png']:
            pattern = os.path.join(rgb_folder, f"*{ext}")
            rgb_files.extend(glob.glob(pattern))
        result['rgb'] = sorted(rgb_files)
    else:
        result['rgb'] = []
    
    # Depth paths
    depth_folder = os.path.join(folder_path, 'depth')
    if os.path.exists(depth_folder):
        depth_files = []
        for ext in ['.png', '.tiff', '.tif']:
            pattern = os.path.join(depth_folder, f"*{ext}")
            depth_files.extend(glob.glob(pattern))
        result['depth'] = sorted(depth_files)
    else:
        result['depth'] = []
    
    # Mask paths
    masks_folder = os.path.join(folder_path, 'masks')
    if os.path.exists(masks_folder):
        mask_files = []
        for ext in ['.png', '.jpg', '.jpeg']:
            pattern = os.path.join(masks_folder, f"*{ext}")
            mask_files.extend(glob.glob(pattern))
        result['masks'] = sorted(mask_files)
    else:
        result['masks'] = []
    
    return result

def wait_for_click_and_get_pixel(image):
    """
    Display an image in a Jupyter notebook and wait for a mouse click.
    Returns (x, y) pixel coordinates of the click.
    """
    coords = []

    def onclick(event):
        # Only respond to left mouse button clicks
        if event.button == 1 and event.xdata is not None and event.ydata is not None:
            coords.append((int(event.xdata), int(event.ydata)))
            plt.close()  # Close the figure after click

    fig, ax = plt.subplots()
    ax.imshow(image)
    ax.set_title("Click on the image")
    cid = fig.canvas.mpl_connect('button_press_event', onclick)

    plt.show()

    if coords:
        return coords[0]
    else:
        return None

def load_camera_param(folder_path: str) -> Dict[str, np.ndarray]:
    """
    Load camera parameters from the specified folder.
    
    Args:
        folder_path (str): Path to the main folder containing /camera_param subdirectory
    
    Returns:
        Dict[str, np.ndarray]: Dictionary with keys 'K', 'D' containing camera parameters
    """
    # load txt
    cam_param_path = os.path.join(folder_path, 'cam_K.txt')
    # if not os.path.exists(camera_param_folder):
    #     raise FileNotFoundError(f"Camera parameter folder not found: {camera_param_folder}")
    
    # load the camera parameters from the camera_param subdirectory
    # with open(cam_param_path, 'r') as file:
    #     lines = [line.strip() for line in file]
    return np.loadtxt(cam_param_path)


In [ ]:
data = load_all_images(data_path)
data['K'] = load_camera_param(data_path)

image_h = data["rgb"][0].shape[0]
image_w = data["rgb"][0].shape[1]

data["K"]


In [ ]:
tapnet = tapir_model.TAPIR(pyramid_level=1, use_casual_conv=True)
tapnet.load_state_dict(torch.load(checkpoint_path))
tapnet = tapnet.to("cuda")
tapnet = tapnet.eval()
torch.set_grad_enabled(False)

In [ ]:
# tapnet related functions
def convert_select_points_to_query_points(frame_id, points):
    """Convert select points to query points.

    Args:
        points: [num_points, 2], [t, y, x]

    Returns:
        query_points: [num_points, 3], [t, y, x]
    """
    points = np.stack(points)
    query_points = np.zeros(shape=(points.shape[0], 3), dtype=np.float32)
    query_points[:, 0] = frame_id
    query_points[:, 1] = points[:, 1] / image_h * resize_h
    query_points[:, 2] = points[:, 0] / image_w * resize_w
    return query_points
    
def convert_query_points_to_image_points(query_points):

    return (transforms.convert_grid_coordinates(
                            copy.deepcopy(query_points).cpu(),
                            (resize_w, resize_h),
                            (image_w, image_h),
                        ).squeeze(0)
                        .squeeze(0)
                        .squeeze(0))


def preprocess_frames( frames):
        """Preprocess frames to model inputs.

        Args:
        frames: [num_frames, height, width, 3], [0, 255], np.uint8

        Returns:
        frames: [num_frames, height, width, 3], [-1, 1], np.float32
        """
        frames = frames.float()
        frames = frames / 255 * 2 - 1
        return frames

def postprocess_occlusions(occlusions, expected_dist):
    visibles = (1 - F.sigmoid(occlusions)) * (1 - F.sigmoid(expected_dist)) > 0.5
    return visibles

def online_model_init(model, frames, query_points):
    """Initialize query features for the query points."""
    frames = preprocess_frames(frames)
    feature_grids = model.get_feature_grids(frames, is_training=False)

    query_features = model.get_query_features(
        frames,
        is_training=False,
        query_points=query_points,
        feature_grids=feature_grids,
    )
    
    return query_features

def online_model_predict(model, frames, query_features, causal_context):
        """Compute point tracks and occlusions given frames and query points."""
        # t0 = time.time()
        frames = preprocess_frames(frames)
        # print("-----------------------------")
        # print("preprocess frame time:", time.time() - t0)
        # t = time.time()
        feature_grids = model.get_feature_grids(frames, is_training=False)
        # print("get_feature_grids time:", time.time() - t)
        # t = time.time()
        trajectories = model.estimate_trajectories(
            frames.shape[-3:-1],
            is_training=False,
            feature_grids=feature_grids,
            query_features=query_features,
            query_points_in_video=None,
            query_chunk_size=64,
            causal_context=causal_context,
            get_causal_context=True,
        )
        # print("estimate_trajectories time:", time.time() - t)
        causal_context = trajectories["causal_context"]
        del trajectories["causal_context"]
        # print("del time: ", time.time() - t)
        # Take only the predictions for the final resolution.
        # For running on higher resolution, it's typically better to average across
        # resolutions.
        tracks = trajectories["tracks"][-1]
        occlusions = trajectories["occlusion"][-1]
        expected_distance = trajectories["expected_dist"][-1]
        uncertainty = copy.deepcopy(F.sigmoid(expected_distance))
        # print("uncertainty: ", F.sigmoid(uncertainty))
        # t = time.time()
        visibles = postprocess_occlusions(occlusions, expected_distance)
        # print("postprocess_occlusions time:", time.time() - t)
        # print("t end: ", time.time() - t0)
        return tracks, uncertainty, visibles, causal_context



In [ ]:
from scipy.spatial import ConvexHull

def check_sample_criterion_uncertainty_threshold(cur_pts, uncertainties, uncertainty_thres = 0.4, ratio = 0.5):
    """
    Check if the current points meet the sampling criterion based on uncertainty threshold.

    Args:
        cur_pts (np.ndarray): Current points of shape (N, 2).
        uncertainties (np.ndarray): Uncertainties associated with the points of shape (N,).
        uncertainty_thres (float): Uncertainty threshold to consider a point as valid.
        ratio (float): Minimum ratio of valid points required to meet the criterion.

    Returns:
        bool: True if we need to resample, False otherwise.
    """
    if len(cur_pts) == 0:
        return True

    valid_mask = uncertainties < uncertainty_thres
    valid_count = np.sum(valid_mask)
    total_count = len(cur_pts)

    if total_count == 0:
        return True

    valid_ratio = valid_count / total_count
    return valid_ratio <= ratio

def check_sample_criterion_area(points: np.ndarray, mask: np.ndarray, threshold: float, mode="convex") -> bool:
    """
    Compute area enclosing points (convex hull or bbox), compare to mask area.

    Args:
        points (np.ndarray): Nx2 array of pixel coordinates (x, y).
        mask (np.ndarray): Binary mask (H, W).
        threshold (float): If area(points)/area(mask) < threshold → True.
        mode (str): "convex" for convex hull area, "bbox" for bounding box area.

    Returns:
        bool
    """
    # --- check mask area ---
    mask_area = np.count_nonzero(mask)
    if mask_area == 0 or points.shape[0] < 3:
        return True  # degenerate case, trivially small

    # --- compute point region area ---
    if mode == "convex":
        try:
            hull = ConvexHull(points)
            point_area = hull.volume  # in 2D, `volume` is the polygon area
        except Exception:
            return True  # convex hull fails (e.g. collinear points)
    elif mode == "bbox":
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)
        point_area = (x_max - x_min + 1) * (y_max - y_min + 1)
    else:
        raise ValueError("mode must be 'convex' or 'bbox'")

    # --- ratio ---
    ratio = point_area / mask_area
    return ratio < threshold

In [ ]:
# def sample_point(model, mask, rgb_image, pose ):
def sample_random_points_in_mask(mask, num_points=10, min_distance=5):
    """
    Sample random points within a binary segmentation mask.
    
    Args:
        mask: Binary mask (H, W) where 1 indicates foreground
        num_points: Number of points to sample
        min_distance: Minimum distance between sampled points (pixels)
    
    Returns:
        points: Array of shape (num_points, 2) with (x, y) coordinates
    """
    import numpy as np
    
    # Find all valid pixel coordinates within the mask
    y_coords, x_coords = np.where(mask > 0)
    
    if len(y_coords) == 0:
        print("Warning: No valid pixels found in mask")
        return np.array([]).reshape(0, 2)
    
    # Convert to (x, y) format
    valid_pixels = np.stack([x_coords, y_coords], axis=1)
    
    if len(valid_pixels) < num_points:
        print(f"Warning: Only {len(valid_pixels)} valid pixels available, returning all")
        return valid_pixels
    
    # Sample points with minimum distance constraint
    sampled_points = []
    attempts = 0
    max_attempts = num_points * 100  # Prevent infinite loops
    
    while len(sampled_points) < num_points and attempts < max_attempts:
        # Randomly select a pixel
        idx = np.random.randint(0, len(valid_pixels))
        candidate = valid_pixels[idx]
        
        # Check if it's far enough from already sampled points
        if len(sampled_points) == 0:
            sampled_points.append(candidate)
        else:
            distances = [np.linalg.norm(candidate - pt) for pt in sampled_points]
            if min(distances) >= min_distance:
                sampled_points.append(candidate)
        
        attempts += 1
    
    if len(sampled_points) < num_points:
        print(f"Warning: Could only sample {len(sampled_points)} points with distance constraint")
    
    return np.array(sampled_points)

In [ ]:
def solve_reg_svd(p,q,init_pose):
    """
    Solve the rigid transformation using SVD method.
    Args:
        p: Nx3 numpy array of 3D points in the source frame
        q: Nx3 numpy array of 3D points in the target frame
        init_pose: 4x4 numpy array of initial pose guess
    Returns:
        R: 3x3 rotation matrix
        t: 3x1 translation vector
    """
    assert p.shape == q.shape and p.shape[1] == 3

    N = p.shape[0]

    # Compute centroids
    centroid_p = np.mean(p, axis=0)
    centroid_q = np.mean(q, axis=0)

    # Center the points
    p_centered = p - centroid_p
    q_centered = q - centroid_q
    
    # Compute covariance matrix
    H = np.dot(p_centered.T, q_centered)
    
    # SVD decomposition
    U, S, Vt = np.linalg.svd(H)
    R = np.dot(Vt.T, U.T)

    # Ensure a proper rotation (det(R) = 1)
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = np.dot(Vt.T, U.T)

    # Compute translation
    t = centroid_q - np.dot(R, centroid_p)

    # cat R and t to form the transformation matrix
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = R
    transformation_matrix[:3, 3] = t
    return transformation_matrix


def solve_reg_svd_refine(
    p: np.ndarray,
    q: np.ndarray,
    init_pose: np.ndarray | None = None,
    max_iters: int = 2,
    inlier_thresh: float | None = None,   # e.g., 0.02 meters; if None, use MAD-based auto threshold
    mad_scale: float = 2.5,               # larger -> keep more points (auto mode only)
    min_inliers: int = 3,
    return_inliers: bool = True,
):
    """
    Robust rigid registration (SVD/Procrustes) with one or more outlier-removal refinements.

    Args:
        p, q: (N,3) source/target with known correspondence (row-wise).
        init_pose: optional (4,4) prior; applied to p before fitting.
        max_iters: total SVD->cull->SVD cycles (>=1).
        inlier_thresh: absolute distance threshold. If None, an automatic MAD-based threshold is used.
        mad_scale: threshold = median(res) + mad_scale * MAD (if inlier_thresh is None).
        min_inliers: minimum inliers required to continue.
        return_inliers: if True, also return the final boolean inlier mask.

    Returns:
        T: (4,4) src->trg transform
        (optional) inliers: (N,) bool mask of points used in the final fit
        (optional) stats: dict with residual stats
    """
    assert p.shape == q.shape and p.shape[1] == 3
    N = p.shape[0]
    if N < 3:
        raise ValueError("Need at least 3 correspondences.")

    def _svd_fit(pa, qa):
        # Pa, Qa: (M,3) centered fit
        cp = pa.mean(axis=0)
        cq = qa.mean(axis=0)
        P = pa - cp
        Q = qa - cq
        H = P.T @ Q
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T
        t = cq - R @ cp
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t
        return T

    def _apply(T, pts):
        pts_h = np.c_[pts, np.ones((pts.shape[0], 1))]
        out = (T @ pts_h.T).T
        return out[:, :3]

    # start with optional prior
    p0 = _apply(init_pose, p) if init_pose is not None else p.copy()

    # initial fit (all points)
    T = _svd_fit(p0, q)

    # iterate: compute residuals, cull, refit
    inliers = np.ones(N, dtype=bool)
    stats = {}
    for it in range(max_iters):
        p_T = _apply(T, p0)
        residuals = np.linalg.norm(p_T - q, axis=1)

        # choose threshold
        if inlier_thresh is None:
            med = np.median(residuals)
            mad = np.median(np.abs(residuals - med)) + 1e-12
            thr = med + mad_scale * 1.4826 * mad  # 1.4826 makes MAD ~ std for Gaussian
        else:
            thr = float(inlier_thresh)

        new_inliers = residuals <= thr

        stats[f"iter_{it}"] = {
            "thr": float(thr),
            "res_median": float(np.median(residuals)),
            "res_mean": float(np.mean(residuals)),
            "res_max": float(np.max(residuals)),
            "num_inliers": int(np.count_nonzero(new_inliers)),
        }

        # stop if no change or too few inliers
        if np.array_equal(new_inliers, inliers) or new_inliers.sum() < min_inliers:
            break

        inliers = new_inliers
        # refit on inliers
        T = _svd_fit(p0[inliers], q[inliers])

    # compose with init_pose to map original p -> q
    if init_pose is not None:
        T = T @ init_pose

    if return_inliers:
        return T, inliers, stats
    return T

In [ ]:
def extract_point_cloud_from_mask(
        mask, rgb_image, depth_image, K, cam2world,
    ):
        """Extract point cloud from masked depth image"""
        # Find valid depth pixels within the mask
        valid_mask = mask & (depth_image > 0)
        if valid_mask.ndim == 3:
            valid_mask = np.squeeze(valid_mask, axis=0)
        y_coords, x_coords = np.where(valid_mask)
        if len(y_coords) == 0:
            return None

        mean_mask_pixel = np.mean(np.stack([x_coords, y_coords], axis=1), axis=0)

        # Extract depth values
        depths = depth_image[y_coords, x_coords]

        # Convert to camera coordinates
        fx = K[0, 0]
        fy = K[1, 1]
        cx = K[0, 2]
        cy = K[1, 2]

        # Convert to camera coordinates
        z = depths / 1000.0
        x = (x_coords - cx) * z / fx
        y = (y_coords - cy) * z / fy

        # Stack coordinates
        points_cam = np.stack([x, y, z], axis=1)

        # Transform to world coordinates
        points_world = (cam2world[:3, :3] @ points_cam.T).T + cam2world[:3, 3]

        # if len(pcd_clean.points) < 10:
        #     return None

        # Create Open3D point cloud
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_world)

        # Add RGB colors if available
        if rgb_image is not None:
            # Extract RGB values for the masked pixels
            rgb_values = rgb_image[y_coords, x_coords]
            # Normalize RGB values to [0, 1] range
            rgb_normalized = rgb_values.astype(np.float64) / 255.0
            pcd.colors = o3d.utility.Vector3dVector(rgb_normalized)

            # Remove outliers
        if pcd_stat_outlier_removal:
            _, ind_stat = pcd.remove_statistical_outlier(
                nb_neighbors=pcd_stat_outlier_removal_nb_neighbors,
                std_ratio=pcd_stat_outlier_removal_std_ratio,
            )

        if pcd_radius_outlier_removal:
            # Distance-based outlier removal if clicked point is provided
            mean_mask_pixel_world = convert_pixel_to_world(
                mean_mask_pixel,
                depth_image,
                K,
                cam2world,
                depth_factor=1000.0,
            )
            # Calculate distances from clicked point to all points
            points_array = points_world
            distances = np.linalg.norm(points_array - mean_mask_pixel_world, axis=1)

            # Keep points within a reasonable distance (e.g., 0.2 meters)
            max_distance = pcd_radius_outlier_removal_radius
            close_indices = np.where(distances <= max_distance)[0]

            # Find the intersection of the two indices (points that pass both filters)
            # ind_stat contains indices from original pcd (statistical filtering)
            # close_indices contains indices from pcd_world_stat (distance filtering)
            # We need to map distance indices back to original point cloud indices
            if pcd_stat_outlier_removal:
                # Both filters applied: find intersection
                ind_union = np.intersect1d(ind_stat, close_indices)
            else:
                # Only distance filter applied
                ind_union = close_indices

            pcd_world_clean = pcd.select_by_index(ind_union)

        else:
            pcd_world_clean = pcd

        # Optional: estimate normals
        # if len(pcd_world_clean.points) > 10:
        #     pcd.estimate_normals(
        #         search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30)
        #     )

        return pcd_world_clean

In [ ]:
def visualize_pcd(pcd):
    # Get points from your point cloud
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    # Create 3D plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot points
    if colors is not None:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c=colors, s=1, alpha=0.8)
    else:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c='blue', s=1, alpha=0.8)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Point Cloud Visualization')
    plt.show()

In [ ]:
# if visualize_pcd_outliers:
#     # Visualize
#     inlier_cloud = pcd.select_by_index(ind_union)
#     inlier_cloud.paint_uniform_color([0, 1, 0])  # Green for inliers
#     outlier_cloud = pcd.select_by_index(ind_union, invert=True)
#     outlier_cloud.paint_uniform_color([1, 0, 0])  # Red for removed points

#     o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])
        

In [ ]:
# initialize the first image and get pcd
rgb_image_first = data["rgb"][0]
depth_image_first = data["depth"][0]
mask = data["masks"][0]
K = data["K"]
cam2world = np.eye(4)

pcd_first = extract_point_cloud_from_mask(mask, rgb_image_first, \
depth_image_first, K, cam2world)

visualize_pcd(pcd_first)

bbox = pcd_first.get_oriented_bounding_box()
# OBB pose (local -> world)
T_local_to_world = np.eye(4, dtype=float)
T_local_to_world[:3, :3] = bbox.R
T_local_to_world[:3, 3] = bbox.center

# Local min/max from extent
half = 0.5 * np.asarray(bbox.extent, dtype=float)
bbox_min_max_local = np.vstack([-half, +half])  # (2,3)

# World -> Cam (self._cam2world is cam->world)
T_world_to_cam = np.linalg.inv(cam2world)
T_local_to_cam = T_world_to_cam @ T_local_to_world

init_pose = T_local_to_cam.copy()
pose = T_local_to_cam

rgb_image_first_vis = rgb_image_first.copy()
vis_img = draw_posed_3d_box(K, rgb_image_first_vis, T_local_to_cam, bbox_min_max_local)
vis_img = draw_xyz_axis(vis_img, T_local_to_cam, K = K)

fig2 = plt.figure(figsize=(12, 8))
ax2 = fig2.add_subplot(111)
ax2.imshow(vis_img)
plt.show()

In [ ]:
def draw_points_on_image(image, points, colors):
    # if points are tensor
    # print(points.shape)
    if isinstance(points, torch.Tensor):
        points = points.cpu().numpy()
        
    for i in range(points.shape[0]):
        cv2.circle(
            image,
            points[i, :].astype(int).reshape(2),
            radius=5,
            color=colors[i],
            thickness=-1,
        )

def get_n_colors(n):
        cmap = plt.get_cmap("RdYlGn")  # or 'tab20', 'jet', etc.
        colors = [tuple(int(c * 255) for c in cmap(i / n)[:3]) for i in range(n)]
        return colors

def get_n_uncertainty_colors(uncertainties, u_min=0.0, u_max=1.0, inverse=False):
        cmap = plt.get_cmap("jet")  # or 'tab20', 'jet', etc.
        norm_uncertainties = (uncertainties - u_min) / (u_max - u_min + 1e-8)
        if inverse:
            norm_uncertainties = 1 - norm_uncertainties
        colors = [tuple(int(c * 255) for c in cmap(u)[:3]) for u in norm_uncertainties]
        return colors
        
def visualize_out_frames_slider(out_frames):
    # Create slider
    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(out_frames)-1,
        step=1,
        description='Frame:',
        continuous_update=False
    )

    # Create output widget
    output = widgets.Output()

    def on_value_change(change):
        with output:
            output.clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            plt.imshow(out_frames[change['new']])
            plt.title(f"Frame {change['new']}")
            plt.axis('off')
            plt.show()

    slider.observe(on_value_change, names='value')

    # Display widgets
    display(slider, output)
    # Trigger initial display
    on_value_change({'new': 0})




def visualize_out_frames_animation(out_frames):

    # Your existing animation code
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title("Image Sequence")

    def animate(frame):
        ax.clear()
        ax.imshow(out_frames[frame])
        ax.set_title(f"Frame {frame}")
        ax.axis('off')
        return ax,

    anim = animation.FuncAnimation(
        fig, animate, frames=len(out_frames), 
        interval=100, repeat=True, blit=False
    )

    display(HTML(anim.to_jshtml()))
    plt.close(fig)

    return anim

In [ ]:


sampled_points = sample_random_points_in_mask(mask, num_points=num_points, min_distance=5)

# choose different colors for each point
point_colors = get_n_colors(len(sampled_points))

# visualize sampled points with colored mask
rgb_image_first_vis2 = rgb_image_first.copy()
draw_points_on_image(rgb_image_first_vis2, sampled_points, point_colors)
plt.imshow(rgb_image_first_vis2)
plt.show()

In [ ]:
def ransac_outlier_filter(
    P: np.ndarray,
    model: str = "center",     # "center" | "plane" | "line"
    thresh: float | None = None,
    max_iters: int = 1000,
    confidence: float = 0.99,  # used to adaptively shrink iterations
    min_inliers: int = 10,
    random_state: int | None = None,
    return_model: bool = False,
):
    """
    RANSAC-based outlier rejection for a single 3D point set.

    Args:
        P: (N,3) array of 3D points.
        model: 'center' (spherical cluster), 'plane', or 'line'.
        thresh: inlier threshold in the same distance units as P.
                If None, a robust MAD-based threshold is computed per hypothesis.
        max_iters: maximum RANSAC iterations.
        confidence: desired success probability (adapts iteration budget as in classic RANSAC).
        min_inliers: minimal inliers for a model to be considered.
        random_state: RNG seed.
        return_model: also return the fitted model parameters.

    Returns:
        inliers: (N,) bool mask of inliers
        (opt) model_params:
            - center: {'c': (3,), 'r_est': float}  (r_est is median inlier radius)
            - plane:  {'n': (3,), 'd': float}      (unit normal n, plane n·x + d = 0)
            - line:   {'p0': (3,), 'v': (3,)}      (point on line, unit direction)
    """
    assert P.ndim == 2 and P.shape[1] == 3
    N = P.shape[0]
    if N < 3:
        raise ValueError("Need at least 3 points.")

    rng = np.random.default_rng(random_state)

    # ---------- model-specific minimal sample sizes and solvers ----------
    if model == "center":
        s = 3  # use 3 points to get a stable centroid; 1 would be too noisy
        def fit_model(idx):
            c = P[idx].mean(axis=0)
            return {"c": c}
        def residuals(params):
            c = params["c"]
            return np.linalg.norm(P - c, axis=1)
        def finalize(params, inliers):
            # robust radius estimate for info
            r_est = float(np.median(np.linalg.norm(P[inliers] - params["c"], axis=1)))
            params["r_est"] = r_est
            return params

    elif model == "plane":
        s = 3
        def fit_model(idx):
            A = P[idx]
            v1 = A[1] - A[0]
            v2 = A[2] - A[0]
            n = np.cross(v1, v2)
            n_norm = np.linalg.norm(n)
            if n_norm < 1e-12:
                return None  # degenerate triplet
            n = n / n_norm
            d = -np.dot(n, A[0])
            return {"n": n, "d": d}
        def residuals(params):
            n, d = params["n"], params["d"]
            return np.abs(P @ n + d)
        def finalize(params, inliers):
            # Optionally refit plane by LS on inliers
            X = P[inliers]
            Xc = X - X.mean(axis=0)
            _, _, vh = np.linalg.svd(Xc, full_matrices=False)
            n = vh[-1, :]
            n /= np.linalg.norm(n)
            d = -np.dot(n, X.mean(axis=0))
            return {"n": n, "d": d}

    elif model == "line":
        s = 2
        def fit_model(idx):
            a, b = P[idx[0]], P[idx[1]]
            v = b - a
            nrm = np.linalg.norm(v)
            if nrm < 1e-12:
                return None
            v = v / nrm
            return {"p0": a, "v": v}
        def residuals(params):
            p0, v = params["p0"], params["v"]
            # distance of points to the line: ||(P - p0) - ((P - p0)·v)v||
            d = P - p0
            proj = (d @ v)[:, None] * v[None, :]
            return np.linalg.norm(d - proj, axis=1)
        def finalize(params, inliers):
            # Refit line via PCA on inliers
            X = P[inliers]
            mu = X.mean(axis=0)
            Xc = X - mu
            _, _, vh = np.linalg.svd(Xc, full_matrices=False)
            v = vh[0, :]
            v /= np.linalg.norm(v)
            return {"p0": mu, "v": v}

    else:
        raise ValueError("model must be 'center', 'plane', or 'line'.")

    # ---------- helpers ----------
    def auto_thresh(res):
        # MAD-based threshold: med + 2.5 * 1.4826 * MAD
        med = np.median(res)
        mad = np.median(np.abs(res - med)) + 1e-12
        return med + 2.5 * 1.4826 * mad

    best_inliers = np.zeros(N, dtype=bool)
    best_params = None
    best_count = 0
    best_median_res = np.inf

    # adaptive iteration count as in RANSAC: k = log(1-p)/log(1-w^s)
    target_iters = max_iters

    it = 0
    while it < min(max_iters, target_iters):
        it += 1
        # sample minimal set
        idx = rng.choice(N, size=s, replace=False)
        params = fit_model(idx)
        if params is None:
            continue
        res = residuals(params)
        thr = auto_thresh(res) if thresh is None else float(thresh)
        inliers = res <= thr
        n_in = int(inliers.sum())
        if n_in >= max(min_inliers, best_count):
            # tie-break on median residual
            med_res = np.median(res[inliers]) if n_in > 0 else np.inf
            if (n_in > best_count) or (n_in == best_count and med_res < best_median_res):
                best_inliers = inliers
                best_params = params
                best_count = n_in
                best_median_res = med_res

                # update adaptive iteration budget
                w = n_in / N
                denom = max(1e-12, 1 - (w ** s))
                target_iters = min(max_iters, int(np.log(1 - confidence) / np.log(denom)) + 1)

    if best_count < min_inliers:
        # fallback: all outliers except the densest small set
        return np.zeros(N, dtype=bool) if not return_model else (np.zeros(N, dtype=bool), None)

    # finalize model on inliers for better params (doesn't change mask)
    best_params = finalize(best_params, best_inliers)

    return (best_inliers, best_params) if return_model else best_inliers

In [ ]:
import numpy as np
import plotly.graph_objects as go

def _apply_tf(points: np.ndarray, tf: np.ndarray) -> np.ndarray:
    """Apply a 4x4 homogeneous transform to Nx3 points."""
    points = np.asarray(points)
    assert points.ndim == 2 and points.shape[1] == 3, "points must be (N,3)"
    assert tf.shape == (4, 4), "tf must be a 4x4 homogeneous matrix"
    homog = np.c_[points, np.ones(len(points))]
    out = homog @ tf.T
    return out[:, :3]

def vis_reg_points(
    src: np.ndarray,
    trg: np.ndarray,
    tf: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    src_color=None,            # None, single color string (e.g. 'red'), or per-point Nx3 in [0,1] or 0..255
    trg_color=None,
    title: str = "Registered point clouds (src→trg)"
):
    """
    Visualize two point clouds after applying tf to src.
    Args:
        src, trg: (N,3) and (M,3) float arrays.
        tf: (4,4) homogeneous transform that maps src -> trg frame.
        backend: 'plotly' (inline, easy) or 'k3d' (very fast for huge clouds).
        point_size: marker size (Plotly) or glyph size (k3d).
        src_color, trg_color: None, single color string/int, or per-point Nx3.
    """
    src_t = _apply_tf(src, tf)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go

        def _to_plotly_color(arr, fallback):
            if arr is None:
                return fallback
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:  # single RGB triplet
                arr = np.tile(arr, (1,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                # normalize if in 0..1
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                else:
                    arr = arr.astype(np.uint8)
                return [f"rgb({r},{g},{b})" for r, g, b in arr]
            # otherwise assume a CSS color string or list Plotly can handle
            return arr

        src_col = _to_plotly_color(src_color, "red")
        trg_col = _to_plotly_color(trg_color, "royalblue")

        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=trg[:,0], y=trg[:,1], z=trg[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=trg_col),
            name="target (trg)"
        ))
        fig.add_trace(go.Scatter3d(
            x=src_t[:,0], y=src_t[:,1], z=src_t[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.85, color=src_col),
            name="source transformed (src·tf)"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
            legend=dict(itemsizing="constant")
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display

        def _to_k3d_colors(arr, n, fallback_hex):
            if arr is None:
                return np.full(n, fallback_hex, dtype=np.uint32)
            arr = np.asarray(arr)
            if arr.ndim == 1 and arr.size == 3:
                arr = np.tile(arr, (n,1))
            if arr.ndim == 2 and arr.shape[1] == 3:
                if arr.max() <= 1.0:
                    arr = (arr * 255).astype(np.uint8)
                arr = arr.astype(np.uint8)
                return ((arr[:,0].astype(np.uint32) << 16) |
                        (arr[:,1].astype(np.uint32) << 8) |
                         arr[:,2].astype(np.uint32))
            # single packed int color or array of packed ints
            return arr.astype(np.uint32)

        trg_pts = trg.astype(np.float32)
        src_pts = src_t.astype(np.float32)
        trg_cols = _to_k3d_colors(trg_color, len(trg_pts), 0x4169E1)  # royalblue
        src_cols = _to_k3d_colors(src_color, len(src_pts), 0xFF0000)  # red

        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(trg_pts, colors=trg_cols, point_size=point_size*0.01, shader='3d', name='trg')
        plot += k3d.points(src_pts, colors=src_cols, point_size=point_size*0.01, shader='3d', name='src·tf')
        plot.camera_auto_fit = True
        display(plot)
    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

def _auto_palette(n: int):
    # deterministic-ish palette in [0..255]
    if n <= 0:
        return np.zeros((0, 3), dtype=np.uint8)
    # golden ratio trick in HSV
    h = (np.arange(n) * 0.61803398875) % 1.0
    s = np.full(n, 0.65)
    v = np.full(n, 0.95)
    # hsv -> rgb
    i = np.floor(h * 6).astype(int)
    f = h * 6 - i
    p = v * (1 - s)
    q = v * (1 - f * s)
    t = v * (1 - (1 - f) * s)
    rgb = np.zeros((n, 3))
    idx = (i % 6 == 0); rgb[idx] = np.stack([v[idx], t[idx], p[idx]], 1)
    idx = (i % 6 == 1); rgb[idx] = np.stack([q[idx], v[idx], p[idx]], 1)
    idx = (i % 6 == 2); rgb[idx] = np.stack([p[idx], v[idx], t[idx]], 1)
    idx = (i % 6 == 3); rgb[idx] = np.stack([p[idx], q[idx], v[idx]], 1)
    idx = (i % 6 == 4); rgb[idx] = np.stack([t[idx], p[idx], v[idx]], 1)
    idx = (i % 6 == 5); rgb[idx] = np.stack([v[idx], p[idx], q[idx]], 1)
    return (rgb * 255).astype(np.uint8)

def _normalize_colors(c, n):
    """
    Accept None, CSS string, single RGB (3,), or Nx3 in [0..1] or [0..255].
    Return:
      - for plotly: list of "rgb(r,g,b)" strings or a single CSS string
      - for k3d: packed uint32 per-point
    """
    if c is None:
        return None
    c = np.asarray(c)
    if c.ndim == 1 and c.size == 3:
        c = np.tile(c[None, :], (n, 1))
    if c.ndim == 2 and c.shape[1] == 3:
        if c.max() <= 1.0:
            c = (c * 255).astype(np.uint8)
        else:
            c = c.astype(np.uint8)
        return c
    # let plotly handle strings/lists of strings; k3d path handles ints
    return c

def _plotly_colorize(c_uint8_or_str, n, fallback):
    if c_uint8_or_str is None:
        return fallback
    if isinstance(c_uint8_or_str, np.ndarray) and c_uint8_or_str.ndim == 2:
        return [f"rgb({r},{g},{b})" for r, g, b in c_uint8_or_str]
    return c_uint8_or_str  # string or list of strings

def _k3d_pack_rgb(c_uint8, n, fallback_hex):
    if c_uint8 is None:
        return np.full(n, fallback_hex, dtype=np.uint32)
    if isinstance(c_uint8, np.ndarray) and c_uint8.ndim == 2:
        c = c_uint8.astype(np.uint32)
        return (c[:,0] << 16) | (c[:,1] << 8) | c[:,2]
    # already an int or array of ints
    return np.asarray(c_uint8, dtype=np.uint32)

# ---------- 1) Single cloud ----------
def vis_point_cloud(
    pts: np.ndarray,
    backend: str = "plotly",
    point_size: float = 2.0,
    color=None,                 # None / str / (3,) / Nx3 in [0..1] or [0..255]
    title: str = "Point Cloud",
):
    assert pts.ndim == 2 and pts.shape[1] == 3
    n = pts.shape[0]
    c = _normalize_colors(color, n)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go
        col = _plotly_colorize(c, n, "royalblue")
        fig = go.Figure()
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col),
            name="cloud"
        ))
        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display
        cols = _k3d_pack_rgb(c, n, 0x4169E1)  # royalblue
        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(pts.astype(np.float32), colors=cols, point_size=point_size*0.01, shader='3d')
        plot.camera_auto_fit = True
        return display(plot)

    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

# ---------- 2) Two clouds (known correspondence, same color per pair) ----------
def vis_corresponded_clouds(
    src: np.ndarray,            # (N,3)
    trg: np.ndarray,            # (N,3) — same order = correspondence
    tf: np.ndarray | None = None,  # optional 4x4 to apply to src before draw
    backend: str = "plotly",
    point_size: float = 2.0,
    palette=None,               # None -> auto palette Nx3; or Nx3 custom colors; or str ignored (auto)
    title: str = "Corresponded clouds (same color = same point)"
):
    assert src.shape == trg.shape and src.shape[1] == 3, "src/trg must be (N,3) with same N"
    N = src.shape[0]
    src_t = _apply_tf(src, tf)

    # build per-point color palette
    if palette is None or (isinstance(palette, str)):
        cols = _auto_palette(N)                     # Nx3 uint8
    else:
        cols = _normalize_colors(palette, N)        # Nx3 uint8
        if not (isinstance(cols, np.ndarray) and cols.ndim == 2 and cols.shape[1] == 3):
            cols = _auto_palette(N)

    if backend.lower() == "plotly":
        import plotly.graph_objects as go
        col_list = [f"rgb({r},{g},{b})" for r, g, b in cols]

        fig = go.Figure()
        # target
        fig.add_trace(go.Scatter3d(
            x=trg[:,0], y=trg[:,1], z=trg[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col_list),
            name="trg (corresponded)"
        ))
        # source (transformed)
        fig.add_trace(go.Scatter3d(
            x=src_t[:,0], y=src_t[:,1], z=src_t[:,2],
            mode="markers",
            marker=dict(size=point_size, opacity=0.9, color=col_list),
            name="src (corresponded)"
        ))

        # (optional) tiny lines between corresponded points
        seg_x = np.vstack([src_t[:,0], trg[:,0], np.full(N, np.nan)]).T.reshape(-1)
        seg_y = np.vstack([src_t[:,1], trg[:,1], np.full(N, np.nan)]).T.reshape(-1)
        seg_z = np.vstack([src_t[:,2], trg[:,2], np.full(N, np.nan)]).T.reshape(-1)
        fig.add_trace(go.Scatter3d(
            x=seg_x, y=seg_y, z=seg_z,
            mode="lines",
            line=dict(width=2),
            name="correspondences",
            showlegend=True
        ))

        fig.update_layout(
            title=title,
            scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
            margin=dict(l=0, r=0, t=40, b=0),
            width=900, height=700,
            legend=dict(itemsizing="constant"),
        )
        return fig.show()

    elif backend.lower() == "k3d":
        import k3d
        from IPython.display import display
        cols_packed = (cols[:,0].astype(np.uint32) << 16) | (cols[:,1].astype(np.uint32) << 8) | cols[:,2].astype(np.uint32)

        plot = k3d.plot(grid_visible=False, height=700)
        plot += k3d.points(trg.astype(np.float32), colors=cols_packed, point_size=point_size*0.01, shader='3d', name='trg')
        plot += k3d.points(src_t.astype(np.float32), colors=cols_packed, point_size=point_size*0.01, shader='3d', name='src')

        # (optional) correspondence segments
        segs = np.stack([src_t, trg], axis=1).astype(np.float32)  # (N,2,3)
        plot += k3d.lines(positions=segs.reshape(-1, 2, 3), colors=cols_packed, width=0.0015)
        plot.camera_auto_fit = True
        return display(plot)

    else:
        raise ValueError("backend must be 'plotly' or 'k3d'")

In [ ]:
from typing import Sequence, Tuple, NamedTuple
import torch


class QueryFeatures(NamedTuple):
  """Query features used to compute trajectories.

  These are sampled from the query frames and are a full descriptor of the
  tracked points. They can be acquired from a query image and then reused in a
  separate video.

  Attributes:
    lowres: Low-resolution features, one for each resolution; each has shape
      [batch, num_query_points, 256]
    hires: High-resolution features, one for each resolution; each has shape
      [batch, num_query_points, 64]
    resolutions: Resolutions used for trajectory computation.  There will be one
      entry for the initialization, and then an entry for each PIPs refinement
      resolution.
  """

  lowres: Sequence[torch.Tensor]
  hires: Sequence[torch.Tensor]
  resolutions: Sequence[Tuple[int, int]]

def concat_query_features(a: QueryFeatures, b: QueryFeatures, point_axis: int = 1) -> QueryFeatures:
    """
    Concatenate two QueryFeatures along the point axis (default 1: [B, N, C]).
    Assumes same number of pyramid levels and identical resolutions.
    """
    assert len(a.lowres) == len(b.lowres) == len(a.hires) == len(b.hires), "Pyramid length mismatch."
    if hasattr(a, "resolutions") and hasattr(b, "resolutions"):
        assert a.resolutions == b.resolutions, "Resolutions must match."

    lowres_cat = [torch.cat([a.lowres[i], b.lowres[i]], dim=point_axis) for i in range(len(a.lowres))]
    hires_cat  = [torch.cat([a.hires[i],  b.hires[i]],  dim=point_axis) for i in range(len(a.hires))]

    # Rebuild the NamedTuple by name
    return QueryFeatures(
        lowres=tuple(lowres_cat),
        hires=tuple(hires_cat),
        resolutions=a.resolutions,  # keep a's (identical to b's)
    )

def infer_num_points_from_qf(qf: QueryFeatures, point_axis: int = 1) -> int:
    """Infer N from any level (use lowres[0] first, else hires[0])."""
    if len(qf.lowres) > 0 and torch.is_tensor(qf.lowres[0]):
        return qf.lowres[0].shape[point_axis]
    if len(qf.hires) > 0 and torch.is_tensor(qf.hires[0]):
        return qf.hires[0].shape[point_axis]
    raise RuntimeError("Could not infer num_query_points from QueryFeatures.")


def pixels_to_resized(points_px, src_hw, dst_wh):
    """(x,y) in original image -> (x',y') in resized image."""
    Hs, Ws = src_hw
    Wd, Hd = dst_wh
    x = points_px[:, 0] * (Wd / Ws)
    y = points_px[:, 1] * (Hd / Hs)
    return np.stack([x, y], axis=1)

def concat_struct(a, b, dim):
    """
    Recursively concat tensors/lists/dicts returned by get_query_features.
    Works when the structure of `a` and `b` matches exactly.
    """
    if torch.is_tensor(a) and torch.is_tensor(b):
        return torch.cat([a, b], dim=dim)
    if isinstance(a, (list, tuple)):
        return type(a)(concat_struct(x, y, dim) for x, y in zip(a, b))
    if isinstance(a, dict):
        return {k: concat_struct(a[k], b[k], dim) for k in a.keys()}
    # fall back: prefer b to avoid silently dropping
    return b

# def expand_causal_state(causal_state, n_new, point_axis=1, seed=None):
#     """
#     Grow every per-point tensor in the causal state by n_new rows along point_axis.
#     Optionally seed some keys from current-frame features (same shapes as the slice to add).
#     """
#     if n_new == 0 or causal_state is None:
#         return causal_state

#     out = []
#     for lvl, st in enumerate(causal_state):
#         new_st = {}
#         for k, v in st.items():
#             if not torch.is_tensor(v):
#                 new_st[k] = v
#                 continue
#             # Assume per-point if it has enough dims and the axis exists
#             if v.dim() > point_axis:
#                 shape = list(v.shape)
#                 shape[point_axis] = n_new
#                 if seed is not None and k in seed and seed[k][lvl] is not None:
#                     add_block = seed[k][lvl]  # must be [*, n_new, *] on point_axis
#                 else:
#                     add_block = torch.zeros(shape, dtype=v.dtype, device=v.device)
#                 new_st[k] = torch.cat([v, add_block], dim=point_axis)
#             else:
#                 new_st[k] = v
#         out.append(new_st)
#     return out
def expand_causal_state_namedtuple(causal_state, n_new: int, point_axis: int = 1):
    """
    Grow every per-point tensor in your causal_state list-of-dicts by n_new along point_axis.
    Leaves non-tensors untouched. Returns a structure with the same nesting.
    """
    if causal_state is None or n_new <= 0:
        return causal_state

    out = []
    for level_dict in causal_state:
        new_level = {}
        for k, v in level_dict.items():
            if torch.is_tensor(v) and v.dim() > point_axis:
                shape = list(v.shape)
                shape[point_axis] = n_new
                pad = torch.zeros(shape, dtype=v.dtype, device=v.device)
                new_level[k] = torch.cat([v, pad], dim=point_axis)
            else:
                new_level[k] = v
        out.append(new_level)
    return out

def initialize_new_points_at_frame(
    frame_idx,
    rgb_image, depth_image, mask,
    tapnet, device,
    resize_w, resize_h,
    image_w, image_h,
    K, cam2world,
    query_features, causal_state, init_point_3d,
    cur_pose,
    pick_points_px=None,          # optional user clicks [(x,y), ...] in ORIGINAL image coords
    max_points_from_mask=20,      # or 0 to disable mask sampling
):
    """
    Returns updated (query_features, causal_state, init_point_3d, new_idx_mask)
    where new_idx_mask is a slice range for the newly-added points.
    """
    # 0) Choose pixels to add (in ORIGINAL image coords)
    if pick_points_px is not None and len(pick_points_px) > 0:
        new_px = np.array(pick_points_px, dtype=np.float32)
    else:
        if max_points_from_mask <= 0:
            return query_features, causal_state, init_point_3d, slice(0,0)
        valid_mask = (mask > 0) & (depth_image > 0)
        ys, xs = np.where(valid_mask)
        if len(xs) == 0:
            return query_features, causal_state, init_point_3d, slice(0,0)
        sel = np.random.choice(len(xs), size=min(max_points_from_mask, len(xs)), replace=False)
        new_px = np.stack([xs[sel], ys[sel]], axis=1).astype(np.float32)

    # 1) Convert to resized pixel coords for TAPIR’s input resolution
    # new_px_resized = pixels_to_resized(new_px, src_hw=(image_h, image_w), dst_wh=(resize_w, resize_h))

    # 2) Compute each new point's **3D anchor** at its start frame (this frame)
    new_p3d, valid_idx = convert_pixel_to_world_batch(
        new_px, depth_image, K, cam2world, depth_factor=1000.0
    )
    new_px = new_px[valid_idx]
    new_p3d        = new_p3d[valid_idx]

    # outlier removal: remove points too far from the object (in local frame)
    # if new_p3d.shape[0] > 0:
    #     dists = torch.norm(new_p3d, dim=1)
    #     inlier_mask = dists < 0.3  # threshold distance (e.g., 0.3 meters)
    #     new_p3d = new_p3d[inlier_mask]
    #     new_px = new_px[inlier_mask]
    inlier_idx = ransac_outlier_filter(new_p3d,thresh=0.6)
    new_p3d = new_p3d[inlier_idx]
    new_px = new_px[inlier_idx]

    if new_px.shape[0] == 0:
        return query_features, causal_state, init_point_3d, slice(0,0)

    # 3) Build TAPIR query points for THIS frame index
    #    (re-use your existing utility so the coordinate convention matches)
    new_query_points = convert_select_points_to_query_points(frame_idx, new_px)  # shape [N_new, 2]
    new_query_points = torch.tensor(new_query_points, dtype=torch.float32, device=device)

    # 4) Extract query features from the CURRENT frame
    rgb_resize = cv2.resize(rgb_image, (resize_w, resize_h))
    frame = torch.tensor(rgb_resize).unsqueeze(0).unsqueeze(0).to(device)  # [1,1,H,W] as in your code
    frames_i = preprocess_frames(frame)
    feature_grids_i = tapnet.get_feature_grids(frames_i, is_training=False)

    new_qf = tapnet.get_query_features(
        frames_i, is_training=False,
        query_points=new_query_points[None],  # [B=1, N_new, 2]
        feature_grids=feature_grids_i,
    )

    # 5) Concatenate query_features along the **point axis** (usually axis=1 for [B,N,...])
    # if query_features is None:
    #     qf_all = new_qf
    #     old_N = 0
    # else:
        # # You may need to adapt `dim=1` if your point axis differs.
        # qf_all = concat_struct(query_features, new_qf, dim=1)
        # # Try to infer old_N from any tensor inside `query_features`
        # sample_tensor = None
        # if torch.is_tensor(query_features):
        #     sample_tensor = query_features
        # elif isinstance(query_features, dict):
        #     sample_tensor = next(v for v in query_features.values() if torch.is_tensor(v))
        # elif isinstance(query_features, (list, tuple)):
        #     # find first tensor inside
        #     def _find_t(x):
        #         if torch.is_tensor(x): return x
        #         if isinstance(x, (list, tuple)):
        #             for xx in x:
        #                 r = _find_t(xx)
        #                 if r is not None: return r
        #         if isinstance(x, dict):
        #             for xx in x.values():
        #                 r = _find_t(xx)
        #                 if r is not None: return r
        #         return None
        #     sample_tensor = _find_t(query_features)
        # old_N = sample_tensor.shape[1] if sample_tensor is not None else 0
        
    # n_new = new_px_resized.shape[0]
    # new_slice = slice(old_N, old_N + n_new)

    # After you computed `new_qf` with tapnet.get_query_features(...)
    if query_features is None:
        qf_all = new_qf
        old_N = 0
    else:
        old_N = infer_num_points_from_qf(query_features, point_axis=1)
        qf_all = concat_query_features(query_features, new_qf, point_axis=1)

    n_new = infer_num_points_from_qf(qf_all, point_axis=1) - old_N

    # Grow causal_state to match new N
    causal_state = expand_causal_state_namedtuple(causal_state, n_new=n_new, point_axis=1)

    # 6) Optionally seed causal_state keys from the current frame features.
    #    If your TAPIR doesn’t expose per-key seeds, just pass seed=None (zero-init).
    # seed = None  # e.g., {"key_memory": [seed_lvl0, seed_lvl1, ...], "value_memory": [...]} each [B, n_new, C]

    # 7) Grow causal_state to match the new total N
    # causal_state = expand_causal_state(causal_state, n_new=n_new, point_axis=1, seed=seed)
    
    # transform the new_p3d to the local frame of the object (init_pose)
    new_p3d = (np.linalg.inv(cur_pose)[:3, :3] @ new_p3d.T).T + np.linalg.inv(cur_pose)[:3, 3]
    
    # 8) Append the per-point metadata you use later
    if init_point_3d is None or init_point_3d.size == 0:
        init_point_3d_all = new_p3d.copy()
    else:
        init_point_3d_all = np.vstack([init_point_3d, new_p3d])

    return qf_all, causal_state, init_point_3d_all

In [ ]:
valid_mask = mask & (depth_image_first > 0)
if valid_mask.ndim == 3:
    valid_mask = np.squeeze(valid_mask, axis=0)
    
# y_coords, x_coords = np.where(valid_mask)
# print(y_coords.shape, x_coords.shape)

# selected_points = np.stack([x_coords, y_coords], axis=1)
# print(selected_points.shape)
# print(sampled_points.shape)
point_3d, valid_idx = convert_pixel_to_world_batch(
    sampled_points,
    depth_image_first,
    K,
    cam2world,
    depth_factor=1000.0,
)

init_point_3d = point_3d[valid_idx].copy()
sampled_points = sampled_points[valid_idx].copy()

# if init_point_3d.shape[0] > 0:
#     dists = np.linalg.norm(init_point_3d, axis=1)
#     inlier_mask = dists < 0.3  # threshold distance (e.g., 0.3 meters)
#     init_point_3d = init_point_3d[inlier_mask]
#     sample_points = sample_points[inlier_mask]
inlier_idx = ransac_outlier_filter(init_point_3d,thresh=0.6)
init_point_3d = init_point_3d[inlier_idx].copy()
sampled_points = sampled_points[inlier_idx]

query_points = convert_select_points_to_query_points(
    0, sampled_points
)
query_points = torch.tensor(query_points).to(device)
# print(query_points.shape)

rgb_resize = cv2.resize(
                    rgb_image_first, (resize_w, resize_h)
                )

frame = torch.tensor(rgb_resize).unsqueeze(0).unsqueeze(0).to(device)

frames = preprocess_frames(frame)
feature_grids = tapnet.get_feature_grids(frames, is_training=False)

query_features = tapnet.get_query_features(
        frames,
        is_training=False,
        query_points=query_points[None],
        feature_grids=feature_grids,
    )

causal_state = tapnet.construct_initial_causal_state(
                        query_points.shape[0],
                        len(query_features.resolutions) - 1,
                    )

with torch.no_grad():
    for i in range(len(causal_state)):
        for k, v in causal_state[i].items():
            causal_state[i][k] = v.to(device)

# print(type(query_features))
# print(query_features.resolutions)
# print(query_features.hires[0][0])
# print(query_features.lowres[0][0])
# print(query_features.hires[0].shape)
# print(query_features.lowres[0].shape)


In [ ]:

out_frames = []
uncertainties = np.ones((len(init_point_3d),), dtype=np.float32)  # start with max uncertainty
points_state = None
uncertainty_thres = 0.6
ratio = 0.5

for i in range(len(data["rgb"])):
    if i == 0:
        continue
    
    rgb_image = data["rgb"][i]
    depth_image = data["depth"][i]
    mask = data["masks"][i]

    rgb_resize = cv2.resize(rgb_image, (resize_w, resize_h))    
    frame = torch.tensor(rgb_resize).unsqueeze(0).unsqueeze(0).to(device)
    
    ind_mask = np.where(mask > 0)

    with torch.no_grad():
        tracks_resize, uncertainty, visibles, causal_state = (
                        online_model_predict(
                            tapnet,
                            frame,
                            query_features,
                            causal_state,
                        )
                    )

        uncertainties = uncertainty.squeeze(0).squeeze(1).cpu().numpy()

        # record max uncertainty
        uncertainty_color = get_n_uncertainty_colors(uncertainty.squeeze(0).squeeze(1).cpu().numpy(),inverse=False)

        tracks = transforms.convert_grid_coordinates(
                            copy.deepcopy(tracks_resize).cpu(),
                            (resize_w, resize_h),
                            (image_w, image_h),
                        )
        
        tracks = tracks.squeeze(0).squeeze(1).cpu().numpy()
        visibles = visibles.squeeze(0).squeeze(1).cpu().numpy().astype(bool)
        # print(visibles.shape)
        # print(tracks.shape)
        # print(init_point_3d.shape)
        tracks_visible = tracks[visibles]
        init_point_3d_visible = init_point_3d[visibles, :]
        uncertainties_visible = uncertainties[visibles]
        uncertainty_color_visible = np.array(uncertainty_color)[visibles]

        points_3d, valid_idx = convert_pixel_to_world_batch(
            tracks_visible,
            depth_image,
            K,
            cam2world,
            depth_factor=1000.0,
        )
        

        # if points_3d[valid_idx].shape[0] < 3:
        #     print(f"Warning: only {points_3d[valid_idx].shape[0]} valid points found, skipping frame {i}")
        #     out_frames.append(rgb_image)
        #     continue
        # cur_pose = solve_reg_svd(init_point_3d_visible[valid_idx], points_3d[valid_idx], pose)
        cur_pose, inliers, _ = solve_reg_svd_refine(init_point_3d_visible[valid_idx], points_3d[valid_idx], pose)
        # print(cur_pose)
        # print(inliers)
        # if check_sample_criterion_threshold_based(init_point_3d, uncertainties, uncertainty_thres=uncertainty_thres, ratio=ratio):
        # if check_sample_criterion_area(tracks_visible, mask, threshold=0.7):
        if i % 3 == 0:
            print(f"Frame {i}: Sampling new points...")
            new_pts = sample_random_points_in_mask(mask, num_points=num_points, min_distance=5)

            # visualize sampled points with colored mask
            rgb_image_first_vis2 = rgb_image.copy()
            draw_points_on_image(rgb_image_first_vis2, new_pts, point_colors)
            # add title frame index
            plt.title(f"Frame {i}: Newly sampled points")
            plt.imshow(rgb_image_first_vis2)
            plt.show()

            query_features, causal_state, init_point_3d = initialize_new_points_at_frame(
                    frame_idx=i,
                    rgb_image=rgb_image,
                    depth_image=depth_image,
                    mask=mask,
                    tapnet=tapnet,
                    device=device,
                    resize_w=resize_w, resize_h=resize_h,
                    image_w=image_w, image_h=image_h,
                    K=K, cam2world=cam2world,
                    query_features=query_features,
                    causal_state=causal_state,
                    init_point_3d=init_point_3d,
                    cur_pose=cur_pose,
                    pick_points_px=new_pts,            # or list of clicks
                    max_points_from_mask=10,        # tune as you like
                )

        if i%15 == 1:
            vis_reg_points(
                init_point_3d_visible[valid_idx],
                points_3d[valid_idx],
                cur_pose,
                backend="plotly",
                point_size=3.0,
                title=f"Frame {i}: Initial alignment (src→trg)",
            )
            vis_corresponded_clouds(
                init_point_3d_visible[valid_idx],
                points_3d[valid_idx],
                tf=cur_pose,
                backend="plotly",
                point_size=3.0,
                title=f"Frame {i}: Corresponded clouds (same color = same point)",
            )

        cur_pose_world = cur_pose @ init_pose 

        

    rgb_out = rgb_image.copy()
    draw_points_on_image(rgb_out, tracks, uncertainty_color)
    rgb_out = draw_posed_3d_box(K, rgb_out, cur_pose_world, bbox_min_max_local)
    rgb_out = draw_xyz_axis(rgb_out, cur_pose_world, K = K)

   
    # draw the points on the image
    out_frames.append(rgb_out)

    

In [ ]:
# visualize init_point_3d as a point cloud
vis_point_cloud(
    init_point_3d,
    backend="plotly",
    point_size=3.0,
    color=None,
    title="All initialized 3D points",
)

In [ ]:
visualize_out_frames_slider(out_frames)

In [ ]:
anim = visualize_out_frames_animation(out_frames)